## Automated Molecular Docking Pipeline using AutoDock Vina

This Python script performs automated molecular docking
of multiple ligands against a target receptor using
AutoDock Vina in Google Colab.

The pipeline is designed for batch docking and
automatically processes all ligand .pdbqt files
stored in the input directory.

Main Features:
--------------
✔ Mounts Google Drive automatically

✔ Detects all ligand .pdbqt files

✔ Performs docking using AutoDock Vina

✔ Saves docked ligand structures (.pdbqt)

✔ Saves docking log files (.txt)

✔ Extracts binding affinity values

✔ Creates CSV summary of docking results

✔ Continues docking even if one ligand fails

✔ Stores all outputs directly in Google Drive

Workflow:
---------
1. Connect Google Drive
2. Load receptor structure
3. Detect ligand files
4. Run AutoDock Vina docking
5. Save docking poses
6. Save docking logs
7. Extract best binding affinity
8. Generate CSV results file

Input Files:
------------
1. Receptor File:
   - 1HSG_receptor.pdbqt

2. Ligand Files:
   - ligand_*_3D.pdbqt

Output Files:
-------------
1. Docked Ligand Structures:
   - *_docked.pdbqt

2. Docking Log Files:
   - *_log.txt

3. Docking Summary CSV:
   - docking_summary.csv

4. Clean Binding Affinity CSV:
   - binding_affinity_results.csv

CSV Output Columns:
-------------------
1. Ligand_ID
2. Binding_Affinity_kcal/mol

Applications:
--------------
✔ Virtual Screening
✔ Drug Discovery
✔ Binding Affinity Analysis
✔ Molecular Interaction Studies
✔ Bioinformatics Research

Software Used:
--------------
✔ AutoDock Vina
✔ Python
✔ Google Colab
✔ Pandas

Author:
-------
Abeera Iftikhar


In [1]:
!apt-get update
!apt-get install -y autodock-vina

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [93.4 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,003 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,915 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,294 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Pac

In [3]:
import os
import glob
import subprocess
import pandas as pd
from google.colab import drive

# =========================================================
# 1. Mount Google Drive
# =========================================================
print("Connecting to Google Drive...")
drive.mount('/content/drive', force_remount=True)

# =========================================================
# 2. Define Paths
# =========================================================
BASE_DIR = "/content/drive/MyDrive/Docking_Pipeline"

INPUT_DIR = os.path.join(BASE_DIR, "prep_data")
OUTPUT_DIR = os.path.join(BASE_DIR, "docking_results")

os.makedirs(OUTPUT_DIR, exist_ok=True)

PDB_ID = "1HSG"

receptor_pdbqt = os.path.join(
    INPUT_DIR,
    f"{PDB_ID}_receptor.pdbqt"
)

# =========================================================
# 3. Grid Box Parameters
# =========================================================
CENTER_X = 16.0
CENTER_Y = 25.0
CENTER_Z = 4.0

SIZE_X = 20.0
SIZE_Y = 20.0
SIZE_Z = 20.0

# =========================================================
# 4. Start Docking
# =========================================================
print("\n--- Starting AutoDock Vina Pipeline ---")

if not os.path.exists(receptor_pdbqt):
    raise FileNotFoundError(
        f"❌ Receptor not found: {receptor_pdbqt}"
    )

# Find ligands
ligand_files = glob.glob(
    os.path.join(INPUT_DIR, "ligand_*_3D.pdbqt")
)

if not ligand_files:
    raise FileNotFoundError(
        "❌ No ligand files found!"
    )

print(f"\nFound {len(ligand_files)} ligands.\n")

# =========================================================
# 5. Store Results
# =========================================================
successful_results = []
failed_results = []

# =========================================================
# 6. Dock Each Ligand
# =========================================================
for lig_pdbqt in ligand_files:

    ligand_name = os.path.basename(
        lig_pdbqt
    ).replace(".pdbqt", "")

    ligand_id = ligand_name.replace(
        "ligand_", ""
    ).replace("_3D", "")

    out_pdbqt = os.path.join(
        OUTPUT_DIR,
        f"{ligand_name}_docked.pdbqt"
    )

    out_log = os.path.join(
        OUTPUT_DIR,
        f"{ligand_name}_log.txt"
    )

    print(f"\n🔹 Docking Ligand ID: {ligand_id}")

    vina_cmd = [
        "vina",
        "--receptor", receptor_pdbqt,
        "--ligand", lig_pdbqt,
        "--center_x", str(CENTER_X),
        "--center_y", str(CENTER_Y),
        "--center_z", str(CENTER_Z),
        "--size_x", str(SIZE_X),
        "--size_y", str(SIZE_Y),
        "--size_z", str(SIZE_Z),
        "--out", out_pdbqt,
        "--exhaustiveness", "8"
    ]

    try:

        result = subprocess.run(
            vina_cmd,
            capture_output=True,
            text=True
        )

        # Save log
        with open(out_log, "w") as f:
            f.write(result.stdout)
            f.write("\n")
            f.write(result.stderr)

        # =====================================================
        # SUCCESS CHECK
        # =====================================================
        if os.path.exists(out_pdbqt):

            affinity = "Not Found"

            # Extract best affinity
            for line in result.stdout.splitlines():

                line = line.strip()

                if line.startswith("1 "):

                    parts = line.split()

                    if len(parts) >= 2:
                        affinity = parts[1]
                        break

            print(f"  ✅ SUCCESS")
            print(f"  🧬 Ligand ID: {ligand_id}")
            print(f"  🔥 Binding Affinity: {affinity} kcal/mol")

            successful_results.append({
                "Ligand_ID": ligand_id,
                "Ligand_Name": ligand_name,
                "Binding_Affinity_kcal/mol": affinity,
                "Docked_File": out_pdbqt,
                "Log_File": out_log
            })

        # =====================================================
        # FAILED DOCKING
        # =====================================================
        else:

            error_msg = result.stderr.strip()

            print(f"  ❌ FAILED")
            print(f"  🧬 Ligand ID: {ligand_id}")
            print(f"  ⚠️ Error: {error_msg}")

            failed_results.append({
                "Ligand_ID": ligand_id,
                "Ligand_Name": ligand_name,
                "Error": error_msg
            })

    except Exception as e:

        print(f"  ❌ EXCEPTION")
        print(f"  🧬 Ligand ID: {ligand_id}")
        print(f"  ⚠️ Exception: {str(e)}")

        failed_results.append({
            "Ligand_ID": ligand_id,
            "Ligand_Name": ligand_name,
            "Error": str(e)
        })

# =========================================================
# 7. Save Successful Docking CSV
# =========================================================
success_csv = os.path.join(
    OUTPUT_DIR,
    "docking_summary.csv"
)

success_df = pd.DataFrame(successful_results)

success_df.to_csv(success_csv, index=False)

# =========================================================
# 8. Save Failed Ligands CSV
# =========================================================
failed_csv = os.path.join(
    OUTPUT_DIR,
    "failed_ligands.csv"
)

failed_df = pd.DataFrame(failed_results)

failed_df.to_csv(failed_csv, index=False)

# =========================================================
# 9. Final Summary
# =========================================================
print("\n=================================================")
print("           DOCKING PIPELINE COMPLETE")
print("=================================================")

print(f"\n✅ Successful Dockings: {len(successful_results)}")
print(f"❌ Failed Dockings: {len(failed_results)}")

print(f"\n✅ Success CSV:")
print(success_csv)

print(f"\n❌ Failed CSV:")
print(failed_csv)

print(f"\n📁 All files saved in:")
print(OUTPUT_DIR)

Connecting to Google Drive...
Mounted at /content/drive

--- Starting AutoDock Vina Pipeline ---

Found 5 ligands.


🔹 Docking Ligand ID: 5281034
  ✅ SUCCESS
  🧬 Ligand ID: 5281034
  🔥 Binding Affinity: -9.865 kcal/mol

🔹 Docking Ligand ID: 2244
  ✅ SUCCESS
  🧬 Ligand ID: 2244
  🔥 Binding Affinity: -5.913 kcal/mol

🔹 Docking Ligand ID: 3672
  ✅ SUCCESS
  🧬 Ligand ID: 3672
  🔥 Binding Affinity: -6.765 kcal/mol

🔹 Docking Ligand ID: 5291
  ✅ SUCCESS
  🧬 Ligand ID: 5291
  🔥 Binding Affinity: -9.338 kcal/mol

🔹 Docking Ligand ID: 123631
  ✅ SUCCESS
  🧬 Ligand ID: 123631
  🔥 Binding Affinity: -9.207 kcal/mol

           DOCKING PIPELINE COMPLETE

✅ Successful Dockings: 5
❌ Failed Dockings: 0

✅ Success CSV:
/content/drive/MyDrive/Docking_Pipeline/docking_results/docking_summary.csv

❌ Failed CSV:
/content/drive/MyDrive/Docking_Pipeline/docking_results/failed_ligands.csv

📁 All files saved in:
/content/drive/MyDrive/Docking_Pipeline/docking_results


In [4]:
import os
import pandas as pd

# =========================================================
# Google Drive Docking Results Folder
# =========================================================
BASE_DIR = "/content/drive/MyDrive/Docking_Pipeline"
OUTPUT_DIR = os.path.join(BASE_DIR, "docking_results")

# =========================================================
# Load Existing Docking Summary
# =========================================================
input_csv = os.path.join(
    OUTPUT_DIR,
    "docking_summary.csv"
)

df = pd.read_csv(input_csv)

# =========================================================
# Keep Only Ligand + Binding Affinity
# =========================================================
results_only = df[[
    "Ligand_ID",
    "Binding_Affinity_kcal/mol"
]]

# =========================================================
# Save Clean CSV
# =========================================================
final_csv = os.path.join(
    OUTPUT_DIR,
    "binding_affinity_results.csv"
)

results_only.to_csv(final_csv, index=False)

# =========================================================
# Display Result
# =========================================================
print("✅ Clean Binding Affinity CSV Saved Successfully!")
print(f"\n📁 File Location:\n{final_csv}")

print("\n🧬 Preview:")
print(results_only.head())

✅ Clean Binding Affinity CSV Saved Successfully!

📁 File Location:
/content/drive/MyDrive/Docking_Pipeline/docking_results/binding_affinity_results.csv

🧬 Preview:
   Ligand_ID  Binding_Affinity_kcal/mol
0    5281034                     -9.865
1       2244                     -5.913
2       3672                     -6.765
3       5291                     -9.338
4     123631                     -9.207
